## 230968174
## Smeet pancholi
### week5
## 9th feb

In [1]:
# Defining the maze, start state, and goal state
maze = [
    [0, 0, 0, 0, 1],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (4, 4)
rows, cols = len(maze), len(maze[0])


In [2]:
# Defining actions, transition model, and heuristic function
def get_neighbors(state):
    moves = [(-1,0), (1,0), (0,-1), (0,1)]
    neighbors = []
    for dx, dy in moves:
        nx, ny = state[0] + dx, state[1] + dy
        if 0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] == 0:
            neighbors.append((nx, ny))
    return neighbors

def manhattan_distance(state):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


## Experiment 4: Recursive Best-First Search (RBFS)

In [3]:
# Implementing Recursive Best First Search (RBFS)
expanded_nodes_rbfs = 0

def rbfs(node, g, f_limit, path, visited):
    global expanded_nodes_rbfs
    expanded_nodes_rbfs += 1

    f = g + manhattan_distance(node)
    if node == goal:
        return path + [node], f

    successors = []
    for neighbor in get_neighbors(node):
        if neighbor not in visited:
            successors.append((neighbor, g + 1))

    if not successors:
        return None, float('inf')

    while True:
        successors.sort(key=lambda x: x[1] + manhattan_distance(x[0]))
        best = successors[0]
        best_f = best[1] + manhattan_distance(best[0])

        if best_f > f_limit:
            return None, best_f

        alternative = successors[1][1] + manhattan_distance(successors[1][0]) if len(successors) > 1 else float('inf')

        result, new_f = rbfs(
            best[0],
            best[1],
            min(f_limit, alternative),
            path + [node],
            visited | {best[0]}
        )

        successors[0] = (best[0], new_f - manhattan_distance(best[0]))

        if result is not None:
            return result, new_f


In [4]:
# Running RBFS and displaying results
expanded_nodes_rbfs = 0
path_rbfs, _ = rbfs(start, 0, float('inf'), [], {start})

print("RBFS Path:", path_rbfs)
print("RBFS Nodes Expanded:", expanded_nodes_rbfs)


RBFS Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
RBFS Nodes Expanded: 9


## Experiment 5: Iterative Deepening A* (IDA*)

In [5]:
# Implementing Iterative Deep# Comparing BFS, Greedy Best First, and A* Search
print("Algorithm Comparison:\n")
print("Algorithm\tPath Length\tNodes Expanded\tOptimal")
print("BFS\t\t", len(bfs_path), "\t\t", bfs_expanded, "\t\tYes")
print("Greedy\t\t", len(greedy_path), "\t\t", greedy_expanded, "\t\tNo")
print("A*\t\t", len(astar_path), "\t\t", astar_expanded, "\t\tYes")
ening A* (IDA*)
expanded_nodes_ida = 0

def ida_star(path, g, threshold, visited):
    global expanded_nodes_ida
    node = path[-1]
    f = g + manhattan_distance(node)

    if f > threshold:
        return f
    if node == goal:
        return path

    minimum = float('inf')
    expanded_nodes_ida += 1

    for neighbor in get_neighbors(node):
        if neighbor not in visited:
            visited.add(neighbor)
            result = ida_star(path + [neighbor], g + 1, threshold, visited)
            if isinstance(result, list):
                return result
            minimum = min(minimum, result)
            visited.remove(neighbor)

    return minimum


In [6]:
# Running IDA* and printing the result
threshold = manhattan_distance(start)
expanded_nodes_ida = 0

while True:
    visited = {start}
    result = ida_star([start], 0, threshold, visited)
    if isinstance(result, list):
        print("IDA* Path:", result)
        print("IDA* Nodes Expanded:", expanded_nodes_ida)
        break
    threshold = result


IDA* Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
IDA* Nodes Expanded: 8


## Experiment 6: Simplified Memory-Bounded A* (SMA*)

In [7]:
# Defining node structure for SMA*
class Node:
    def __init__(self, state, g, parent=None):
        self.state = state
        self.g = g
        self.h = manhattan_distance(state)
        self.f = self.g + self.h
        self.parent = parent


In [8]:
# Implementing Simplified Memory Bounded A* (SMA*)
expanded_nodes_sma = 0

def sma_star(memory_limit):
    global expanded_nodes_sma
    open_list = [Node(start, 0)]
    visited = set()

    while open_list:
        open_list.sort(key=lambda n: n.f)
        current = open_list.pop(0)
        expanded_nodes_sma += 1

        if current.state == goal:
            path = []
            while current:
                path.append(current.state)
                current = current.parent
            return path[::-1]

        visited.add(current.state)

        for neighbor in get_neighbors(current.state):
            if neighbor not in visited:
                child = Node(neighbor, current.g + 1, current)
                open_list.append(child)

        if len(open_list) > memory_limit:
            open_list.sort(key=lambda n: n.f, reverse=True)
            open_list.pop(0)

    return None


In [9]:
# Running SMA* with fixed memory limit
expanded_nodes_sma = 0
path_sma = sma_star(memory_limit=10)

print("SMA* Path:", path_sma)
print("SMA* Nodes Expanded:", expanded_nodes_sma)


SMA* Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
SMA* Nodes Expanded: 10
